# CNNs with Keras on landscape data

## Dataset download from git

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-landscape.git
!ls dataset-landscape
print("***")
!ls -l dataset-landscape/seg_train
print("***")
!ls -l dataset-landscape/seg_pred

## Imports

In [ ]:
!pip install tf-keras
import itertools
import os
import pathlib
import random
import typing

import matplotlib
import matplotlib.pyplot as plt
import numpy
import PIL
import seaborn
import sklearn.metrics
import tensorflow as tf
import tf_keras as keras
import tqdm.notebook

## Data preparation

In [ ]:
INPUT_SHAPE = (150, 150, 3)


label_names = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
label_to_index = {l: i for i, l in enumerate(label_names)}


def get_images(dir_path: pathlib.Path,
               size: int = INPUT_SHAPE[0],
               channels_first: bool = False,
               shuffle: bool = True,
               create_labels: bool = True,
               ) -> typing.Tuple[tf.Tensor, tf.Tensor]:
  images = []
  if create_labels:
    labels = []

  for subdir_path in tqdm.notebook.tqdm(
      list(dir_path.iterdir()), desc="Processing folders"):

    dir_name = subdir_path.name

    if create_labels:
      label = label_to_index.get(dir_name)

    for image_path in tqdm.notebook.tqdm(
        list(subdir_path.iterdir()), desc=f"Folder {dir_name}", leave=False):
      images.append(
          numpy.array(PIL.Image.open(image_path).resize((size, size))))
      if create_labels:
        labels.append(label)

  images = tf.constant(numpy.array(images))
  if create_labels:
    labels = tf.constant(numpy.array(labels))

  if shuffle:
    perm = tf.random.shuffle(tf.range(images.shape[0]))
    images = tf.gather(images, perm)
    if create_labels:
      labels = tf.gather(labels, perm)

  if channels_first:
    images = tf.transpose(images, perm=(0, 3, 1, 2))

  if create_labels:
    return images, labels
  else:
    return images

## Process the data with `get_images`

In [ ]:
images, labels = get_images(
    pathlib.Path("dataset-landscape") / "seg_train")

In [ ]:
print(f"Images shape: {images.shape}")
print(f"Labels shape: {labels.shape}")
seaborn.countplot(x=labels)
plt.title("Label counts")
plt.ylabel("Count")
plt.xlabel("Label")
plt.show()

In [ ]:
f, ax = plt.subplots(5, 5, figsize=(15, 15))

random_indexes = numpy.random.choice(images.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    image = images[img_index]
    label = label_names[labels[img_index]]

    ax[i, j].imshow(image)
    ax[i, j].set_title(f"Example {img_index} ({label})")
    ax[i, j].axis('off')

## Model design

First, a “minimalistic” CNN

In [ ]:
model = keras.models.Sequential()
model.add(keras.layers.Input((150, 150, 3)))
model.add(keras.layers.Conv2D(1,
                              kernel_size=(3, 3),
                              activation="relu"))
model.add(keras.layers.MaxPool2D(3,3))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(6, activation="softmax"))
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

## Explaining the number of parameters

Convolution layer:

\begin{alignat}{5}
  &\text{kernel_size} &&\times \text{n_kernels} &&\times \text{n_channels} &&+ \text{n_biases} \\
  ={ } &(3 \times 3) &&\times 1 &&\times 3 &&+ 1
\end{alignat}

Dense layer:

\begin{alignat}{4}
  &\text{input_size} &&\times \text{output_size} &&+ \text{n_biases} \\
  ={ } &(49 \times 49) &&\times 6 &&+ 6 \\
  ={ } &2401 &&\times 6 &&+ 6
\end{alignat}

## Training

For now, let's run the training for one single epoch (full pass of the data).

In [ ]:
training_history = model.fit(images, labels, epochs=1, validation_split=0.30)

## Improving performance

In [ ]:
conv2d_params = dict(kernel_size=(3,3),
                     activation="relu",
                     kernel_initializer="orthogonal",
                     padding="same")

dense_params = dict(activation="relu", kernel_initializer="orthogonal")

model = keras.models.Sequential()
model.add(keras.layers.Input((150, 150, 3)))
model.add(keras.layers.Conv2D(200, **conv2d_params))
model.add(keras.layers.MaxPool2D(2, 2, padding="same"))
model.add(keras.layers.Conv2D(200, **conv2d_params))
model.add(keras.layers.MaxPool2D(2, 2, padding="same"))
model.add(keras.layers.Conv2D(200, **conv2d_params))
model.add(keras.layers.MaxPool2D(2, 2, padding="same"))
model.add(keras.layers.Conv2D(200, **conv2d_params))
model.add(keras.layers.MaxPool2D(2, 2, padding="same"))
model.add(keras.layers.Conv2D(200, **conv2d_params))
model.add(keras.layers.MaxPool2D(2, 2, padding="same"))
model.add(keras.layers.Conv2D(200, **conv2d_params))
model.add(keras.layers.MaxPool2D(2, 2, padding="same"))
model.add(keras.layers.Conv2D(200, **conv2d_params))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dropout(rate=0.7))
model.add(keras.layers.Dense(200, **dense_params))
model.add(keras.layers.Dense(100, **dense_params))
model.add(keras.layers.Dense(50, **dense_params))
model.add(keras.layers.Dropout(rate=0.2))
model.add(keras.layers.Dense(6,
                             activation="softmax",
                             kernel_initializer="orthogonal"))

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

In [ ]:
training = model.fit(images,
                     labels,
                     epochs=15,
                     validation_split=0.30,
                     batch_size=128)

def plot_metrics(history) -> None:
  plt.plot(training.history["accuracy"])
  plt.plot(training.history["val_accuracy"])
  plt.title("Model accuracy")
  plt.ylabel("Accuracy")
  plt.xlabel("Epoch")
  plt.legend(["Training", "Validation"], loc="upper left")
  plt.show()

  plt.plot(training.history["loss"])
  plt.plot(training.history["val_loss"])
  plt.title("Model loss")
  plt.ylabel("Loss")
  plt.xlabel("Epoch")
  plt.legend(["Training", "Validation"], loc="upper right")
  plt.show()

# Training metrics visualization
plot_metrics(training.history)

## Performance evaluation on the test set

Let's evaluate our model on unseen data.

In [ ]:
test_images,test_labels = get_images(
    pathlib.Path("dataset-landscape") / "seg_test")
model.evaluate(test_images, test_labels, verbose=1)

## Error analysis

We can now look at the confusion matrix and at some misclassified examples.

In [ ]:
test_pred = numpy.argmax(model.predict(test_images), axis=-1)
confusion_matrix = sklearn.metrics.confusion_matrix(test_pred,
                                                    test_labels)
seaborn.heatmap(confusion_matrix,
                  cmap="rocket_r",
                  xticklabels=label_names,
                  yticklabels=label_names,
                  annot=True,
                  fmt="d")
plt.title("Confusion matrix")
plt.show()

seaborn.countplot(x=test_labels)
plt.title("Predicted class counts")
plt.ylabel("Count")
plt.xlabel("Predicted class")
plt.show()

In [ ]:
def plot_mistakes(predicted_class: str, true_class: str) -> None:
  mistakes = test_images[(test_pred == label_to_index[predicted_class])
                         & (test_labels == label_to_index[true_class])]
  random_indexes = numpy.random.choice(mistakes.shape[0],
                                       size=min(mistakes.shape[0], 25),
                                       replace=False)
  grid_indexes = itertools.product(range(5), repeat=2)

  f, ax = plt.subplots(5, 5, figsize=(15, 15))
  for img_index, (i, j) in zip(random_indexes, grid_indexes):
    ax[i, j].imshow(mistakes[img_index])
    ax[i, j].axis("off")

In [ ]:
# Plot images predicted as glacier when they were actually mountain
plot_mistakes("glacier", "mountain")

In [ ]:
# Plot images predicted as glacier when they were actually sea
plot_mistakes("glacier", "sea")

In [ ]:
# Plot images predicted as buildings when they were actually sea
plot_mistakes("buildings", "sea")

## Transfer learning

In [ ]:
base_model = keras.applications.EfficientNetB0(include_top=False,
                                               weights="imagenet",
                                               input_shape=(150, 150, 3))

base_model.trainable = False

model = keras.Sequential(
    [base_model,
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(1024, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(256, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(64, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(16, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Flatten(),
     keras.layers.Dense(6, activation="softmax", kernel_regularizer="l2")])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

In [ ]:
training = model.fit(images,
                     labels,
                     epochs=5,
                     validation_split=0.30,
                     batch_size=512)

plot_metrics(training.history)

## Predict in “real” conditions

We'll now use data in `seg_pred`: we have no label for them.

The best we can do is display the full probability distribution that our model computes.

In [ ]:
pred_images = get_images(
    pathlib.Path("dataset-landscape") / "seg_pred",
    create_labels=False)

In [ ]:
f, ax = plt.subplots(10, 5, figsize=(30, 45))

random_indexes = numpy.random.choice(pred_images.shape[0],
                                     size=25,
                                     replace=False)

grid_indexes = itertools.product(range(0, 10, 2), range(5))

for img_index, (i, j) in zip(random_indexes, grid_indexes):
  image = pred_images[img_index]
  probabilities = model.predict(image[None, ...])[0]
  predicted_class = label_names[numpy.argmax(probabilities)]

  ax[i, j].imshow(image)
  ax[i, j].set_title(f"Exemple {img_index}")
  ax[i, j].axis('off')

  ax[i + 1, j].bar(label_names, probabilities)